In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pickle
import tensorflow as tf
import keras
from keras import ops
import random

from sklearn.preprocessing import MinMaxScaler
from robot_vlp.config import INTERIM_DATA_DIR, PROCESSED_DATA_DIR, TRAINING_LOGS_DIR, MODELS_DIR

import robot_vlp.data.preprocessing as p
import robot_vlp.modeling.rnn_hyperparameter_search as hps
import robot_vlp.modeling.rnn as rnn

%load_ext autoreload
%autoreload 2


2024-12-18 11:00:02.502 | INFO     | robot_vlp.config:<module>:11 - PROJ_ROOT path is: /Users/tyrelglass/PhD/Repositories/robot-vlp


In [2]:
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

In [3]:
tf.config.list_physical_devices()

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'),
 PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [7]:
tuner = hps.build_random_search_tuner(TRAINING_LOGS_DIR, 'odo_nav_rnd_search', overwrite = False)

Reloading Tuner from /Users/tyrelglass/PhD/Repositories/robot-vlp/models/training_logs/odo_nav_rnd_search/tuner0.json


In [8]:
tensorboard_cb = tf.keras.callbacks.TensorBoard(TRAINING_LOGS_DIR / 'odo_nav_rnd_search/tensorboard')
early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=3)

In [4]:
tf.config.set_visible_devices([], 'GPU')

In [5]:
with open(PROCESSED_DATA_DIR/'odometer_navigated_data.pkl', 'rb') as handle:
    data = pickle.load(handle)

In [6]:
data['X_test'].shape

(133365, 40, 6)

In [25]:
tuner.search(
    x = data['X_valid'][:50000],
    y = [data['y_valid'][:50000,[0,1]],  p.ang_to_vector(data['y_valid'][:50000,2], unit = 'degrees').numpy()],
    epochs = 20,
    validation_data = (data['X_test'][:], [data['y_test'][:,[0,1]], p.ang_to_vector(data['y_test'][:,2], unit = 'degrees').numpy()]), 
    callbacks = [early_stopping_cb, tensorboard_cb],
    batch_size = 1024
    )

Trial 48 Complete [00h 06m 03s]
val_loss: 2.1961257457733154

Best val_loss So Far: 0.06932250410318375
Total elapsed time: 13h 51m 42s

Search: Running Trial #49

Value             |Best Value So Far |Hyperparameter
7                 |4                 |n_hidden
185               |200               |n_neurons
0.0005            |0.0021            |learning_rate
sgd               |adam              |optimizer
lstm              |gru               |layer_type

Epoch 1/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 174s 4s/step - heading_loss: 0.9609 - loss: 5.7981 - pos_loss: 4.8369 - val_heading_loss: 0.9639 - val_loss: 2.1934 - val_pos_loss: 1.2252
Epoch 2/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 172s 4s/step - heading_loss: 0.9681 - loss: 2.1833 - pos_loss: 1.2153 - val_heading_loss: 0.9633 - val_loss: 2.1884 - val_pos_loss: 1.2208
Epoch 3/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 172s 4s/step - heading_loss: 0.9606 - loss: 2.1792 - pos_loss: 1.2186 - val_heading_loss: 0.9631 - val_loss: 2.1878 - val_pos_loss: 1.2205
Epoch 4

KeyboardInterrupt: 

In [29]:
model = tuner.get_best_models()[0]

/Users/tyrelglass/miniforge3/envs/robot-vlp/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:713: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 34 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [31]:
model.save(MODELS_DIR / 'odometer_trained_rnn.keras')

In [13]:
model = keras.models.load_model(MODELS_DIR / 'odometer_trained_rnn.keras',custom_objects={"ang_loss_fn": rnn.ang_loss_fn})



/Users/tyrelglass/miniforge3/envs/robot-vlp/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:719: UserWarning: Skipping variable loading for optimizer 'adam', because it has 34 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [2]:
tf.config.set_visible_devices([], 'GPU')

In [3]:
#takes 4 hours with gpu
#takes 25 min with cpu

rnn.retrain_model('odometer_trained_rnn.keras', 'odometer_navigated_data.pkl')

Epoch 1/2000


/Users/tyrelglass/miniforge3/envs/robot-vlp/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:719: UserWarning: Skipping variable loading for optimizer 'adam', because it has 34 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


  149/12237 ━━━━━━━━━━━━━━━━━━━━ 27:57 139ms/step - heading_loss: 0.2721 - loss: 0.7448 - pos_loss: 0.4727

KeyboardInterrupt: 

In [ ]:
%load_ext tensorboard
%tensorboard --logdir=./my_logs